# Predictive Performance

In [1]:
import re
import datasets
import pandas as pd
import numpy as np
import statsmodels.api as sm

from lightgbm import LGBMRegressor
from sklearn.metrics import r2_score

from utils.utils_data2 import (
    load_pred_and_emb,
    center_and_norm,
    get_pca,
    get_cluster,
    get_similarities,
    add_lags_and_scale_data,
)

In [2]:
txt_only = False
embeddings = True
dataframe_name = f"dataset_txt_only_{txt_only}_embeddings_{embeddings}"
df_full_train = pd.read_csv(f"../../data/A4_replication_clothes_data/{dataframe_name}_train.zip")
df_full_val = pd.read_csv(f"../../data/A4_replication_clothes_data/{dataframe_name}_val.zip")

columns_to_drop = [
    "Q_t-2",
    "P_bb_t-2",
    "REVIEW_COUNT_t-2",
    "RATING_t-2",
    # 'pred_ml_l_diff_lag_2',
    # 'pred_ml_m_diff_lag_2',
]
df_full_train = df_full_train.drop(columns=columns_to_drop)
df_full_val = df_full_val.drop(columns=columns_to_drop)

df_full_train = df_full_train.dropna()
df_full_val = df_full_val.dropna()

dummy_subcat_names = [
    category for category in df_full_val["subcat_aggregated"].unique()
]
all_time_steps = sorted([str(date) for date in df_full_val["date"].unique()])

print(f"Dummy subcat names: {dummy_subcat_names}")
print(f"All time steps: {all_time_steps}")
df_full_val.columns

Dummy subcat names: ['Residual', 'T-Shirts', 'Casual', 'Active Shorts', 'Sunglasses', 'Tunics', 'Fashion Sneakers', 'Everyday Bras', 'Cardigans', 'Sports Bras', 'Tanks & Camis', 'Pullovers', 'Briefs', 'Button-Down Shirts', 'Cover-Ups', 'Women, Skirts', 'Slippers', 'Leggings', 'Sets', 'Jeans', 'Blouses & Button-Down Shirts']
All time steps: ['2023-02-27', '2023-03-27', '2023-04-24', '2023-05-22', '2023-06-19', '2023-07-17', '2023-08-14', '2023-09-11', '2023-10-09', '2023-11-06', '2023-12-04', '2024-01-01', '2024-01-29', '2024-02-26', '2024-03-25']


Index(['ASIN', 'date', 'index', 'Q_t', 'P_bb_t', 'text', 'window',
       'REVIEW_COUNT', 'RATING', 'Anzahl Neu Angebote: Aktuell',
       ...
       'Q_t-1', 'P_bb_t-1', 'REVIEW_COUNT_t-1', 'RATING_t-1', 'Delta_Q_t',
       'Delta_P_bb_t', 'pred_ml_l', 'pred_ml_m', 'pred_ml_l_lag_1',
       'pred_ml_m_lag_1'],
      dtype='object', length=349)

In [3]:
print(len(df_full_val))

13734


In [4]:
print(len(df_full_train))

13599


## Variable Names

In [5]:
n_lags = 1

outcome = "Q_t"
treatment = "P_bb_t"

outcome_diff = "Delta_Q_t"
treatment_diff = "Delta_P_bb_t"

dummy_time_steps = all_time_steps[n_lags:]

all_dummy_controls = (
    dummy_subcat_names
    + dummy_time_steps
    + [
        "Blitzangebot: Ist Tiefstpreis",
    ]
)

dummy_baselines = [dummy_time_steps[0], "Residual"]
dummy_time_steps_wo_baseline = [t for t in dummy_time_steps if t not in dummy_baselines]
dummy_controls = [t for t in all_dummy_controls if t not in dummy_baselines]

cont_controls = [
    "RATING_t-1",
    "REVIEW_COUNT_t-1",
    "Anzahl Neu Angebote: Aktuell",
    "Anzahl der abgerufenen Live-Angebote: Neu, FBA",
    "Anzahl der abgerufenen Live-Angebote: Neu, FBM",
]

add_controls_to_scale = [
    "RATING",
    "REVIEW_COUNT",
]

controls_pca = ["pca_0", "pca_1", "pca_2", "pca_3", "pca_4"]
controls_similarities = [
    "similarity_cluster_0",
    "similarity_cluster_1",
    "similarity_cluster_2",
    "similarity_cluster_3",
    "similarity_cluster_4",
]

additional_controls = cont_controls + dummy_controls

additional_controls_deep = [
    var for var in additional_controls if var not in dummy_subcat_names
]

controls_emb = [var for var in df_full_train.columns if "emb" in var]

In [6]:
# dataframe for all results
column_names = ["R2 Q Train", "R2 Q Test", "R2 P Train", "R2 P Test"]

results_df = pd.DataFrame(columns=column_names)
results_df_diff = pd.DataFrame(columns=column_names)

## Performance Deep Model with time independent controls

In [7]:
df_dict = {
    "Train": df_full_train,
    "Test": df_full_val,
}

results_df_deep = pd.DataFrame(
    data=np.full((2, 4), np.nan),
    columns=column_names,
    index=["Deep Time Independent", "Deep Time Dependent"],
)

for df_name, df in df_dict.items():
    y = df[outcome].values
    d = df[treatment].values

    pred_ml_l = df["pred_ml_l"].values
    pred_ml_m = df["pred_ml_m"].values

    r2_ml_l = np.round(r2_score(y, pred_ml_l), 4)
    r2_ml_m = np.round(r2_score(d, pred_ml_m), 4)

    print(f"Evaluation for {df_name} set")
    print(f"R2 Outcome: {r2_ml_l}")
    print(f"R2 Treatment: {r2_ml_m}")

    pred_ml_l_lag1 = df["pred_ml_l_lag_1"].values
    pred_ml_m_lag1 = df["pred_ml_m_lag_1"].values

    r2_ml_l_lag1 = np.round(r2_score(y, pred_ml_l_lag1), 4)
    r2_ml_m_lag1 = np.round(r2_score(d, pred_ml_m_lag1), 4)

    print(f"Evaluation for {df_name} set")
    print(f"R2 Outcome (time dependent): {r2_ml_l_lag1}")
    print(f"R2 Treatment (time dependent): {r2_ml_m_lag1}")

    results_df_deep[f"R2 Q {df_name}"] = (r2_ml_l, r2_ml_l_lag1)
    results_df_deep[f"R2 P {df_name}"] = (r2_ml_m, r2_ml_m_lag1)

results_df_deep

Evaluation for Train set
R2 Outcome: 0.6725
R2 Treatment: 0.9639
Evaluation for Train set
R2 Outcome (time dependent): 0.703
R2 Treatment (time dependent): 0.9626
Evaluation for Test set
R2 Outcome: 0.1492
R2 Treatment: 0.6644
Evaluation for Test set
R2 Outcome (time dependent): 0.3328
R2 Treatment (time dependent): 0.634


,R2 Q Train,R2 Q Test,R2 P Train,R2 P Test
Deep Time Independent,0.6725,0.1492,0.9639,0.6644
Deep Time Dependent,0.7030,0.3328,0.9626,0.6340


## Feature generation

In [11]:
# train set
y_train = df_full_train[outcome].squeeze()
d_train = df_full_train[treatment].squeeze()
y_train_diff = df_full_train[outcome_diff].squeeze()
d_train_diff = df_full_train[treatment_diff].squeeze()

x_train = df_full_train[additional_controls]
x_train = sm.add_constant(x_train)

x_train_pca = df_full_train[additional_controls + controls_pca]
x_train_pca = sm.add_constant(x_train_pca)
x_train_sim = df_full_train[additional_controls + controls_similarities]
x_train_sim = sm.add_constant(x_train_sim)
x_train_emb = df_full_train[additional_controls + controls_emb]
x_train_emb = sm.add_constant(x_train_emb)

# test set
y_test = df_full_val[outcome].squeeze()
d_test = df_full_val[treatment].squeeze()
y_test_diff = df_full_val[outcome_diff].squeeze()
d_test_diff = df_full_val[treatment_diff].squeeze()

x_test = df_full_val[additional_controls]
x_test = sm.add_constant(x_test)

x_test_pca = df_full_val[additional_controls + controls_pca]
x_test_pca = sm.add_constant(x_test_pca)
x_test_sim = df_full_val[additional_controls + controls_similarities]
x_test_sim = sm.add_constant(x_test_sim)
x_test_emb = df_full_val[additional_controls + controls_emb]
x_test_emb = sm.add_constant(x_test_emb)

# update names for boosting
x_train.rename(columns=lambda x: re.sub("[^A-Za-z0-9_]+", "", x), inplace=True)
x_test.rename(columns=lambda x: re.sub("[^A-Za-z0-9_]+", "", x), inplace=True)

x_train_pca.rename(columns=lambda x: re.sub("[^A-Za-z0-9_]+", "", x), inplace=True)
x_test_pca.rename(columns=lambda x: re.sub("[^A-Za-z0-9_]+", "", x), inplace=True)

x_train_sim.rename(columns=lambda x: re.sub("[^A-Za-z0-9_]+", "", x), inplace=True)
x_test_sim.rename(columns=lambda x: re.sub("[^A-Za-z0-9_]+", "", x), inplace=True)

x_train_emb.rename(columns=lambda x: re.sub("[^A-Za-z0-9_]+", "", x), inplace=True)
x_test_emb.rename(columns=lambda x: re.sub("[^A-Za-z0-9_]+", "", x), inplace=True)

In [12]:
train_dict = {
    "y": y_train,
    "y_diff": y_train_diff,
    "d": d_train,
    "d_diff": d_train_diff,
    "x": x_train,
    "x_emb": x_train_emb,
    "x_pca": x_train_pca,
    "x_sim": x_train_sim,
}

test_dict = {
    "y": y_test,
    "y_diff": y_test_diff,
    "d": d_test,
    "d_diff": d_test_diff,
    "x": x_test,
    "x_emb": x_test_emb,
    "x_pca": x_test_pca,
    "x_sim": x_test_sim,
}

## Performance Level Data

In [13]:
feature_specifications = ["x", "x_pca", "x_sim", "x_emb"]
results_df_tab = pd.DataFrame()

for feature_specification in feature_specifications:
    print("\n ------------------------------------")
    print(f"Feature specification: {feature_specification}")

    print("\nOLS")
    outcome_reg = sm.OLS(train_dict["y"], train_dict[feature_specification]).fit()
    ols_outcome_train = r2_score(
        train_dict["y"], outcome_reg.predict(train_dict[feature_specification])
    )
    print(f"R2 Outcome (train): {ols_outcome_train}")
    ols_outcome_test = r2_score(
        test_dict["y"], outcome_reg.predict(test_dict[feature_specification])
    )
    print(f"R2 Outcome (test): {ols_outcome_test}")

    treatment_reg = sm.OLS(train_dict["d"], train_dict[feature_specification]).fit()
    ols_treatment_train = r2_score(
        train_dict["d"], treatment_reg.predict(train_dict[feature_specification])
    )
    print(f"R2 Treatment (train): {ols_treatment_train}")
    ols_treatment_test = r2_score(
        test_dict["d"], treatment_reg.predict(test_dict[feature_specification])
    )
    print(f"R2 Treatment (test): {ols_treatment_test}")

    print("\nBoosting")
    boost_q_outcome = LGBMRegressor(
        n_estimators=500, lr=0.02, random_state=42, verbose=-1
    )
    boost_q_outcome.fit(train_dict[feature_specification], train_dict["y"])
    r2_boost_q_train = boost_q_outcome.score(
        train_dict[feature_specification], train_dict["y"]
    )
    print(f"R2 Outcome (train): {r2_boost_q_train}")
    r2_boost_q_test = boost_q_outcome.score(
        test_dict[feature_specification], test_dict["y"]
    )
    print(f"R2 Outcome (test): {r2_boost_q_test}")

    boost_q_treatment = LGBMRegressor(
        n_estimators=500, lr=0.02, random_state=42, verbose=-1
    )
    boost_q_treatment.fit(train_dict[feature_specification], train_dict["d"])
    r2_boost_d_train = boost_q_treatment.score(
        train_dict[feature_specification], train_dict["d"]
    )
    print(f"R2 Treatment (train): {r2_boost_d_train}")
    r2_boost_d_test = boost_q_treatment.score(
        test_dict[feature_specification], test_dict["d"]
    )
    print(f"R2 Treatment (test): {r2_boost_d_test}")

    results_df_tab_tmp = pd.DataFrame(
        data=[
            [
                ols_outcome_train,
                ols_outcome_test,
                ols_treatment_train,
                ols_treatment_test,
            ],
            [r2_boost_q_train, r2_boost_q_test, r2_boost_d_train, r2_boost_d_test],
        ],
    )
    results_df_tab = pd.concat([results_df_tab, results_df_tab_tmp], axis=0)

row_names = [
    "OLS (Tabular)",
    "Boosting (Tabular)",
    "OLS (Tabular + PCA)",
    "Boosting (Tabular + PCA)",
    "OLS (Tabular + Similarities)",
    "Boosting (Tabular + Similarities)",
    "OLS (Tabular + Embeddings)",
    "Boosting (Tabular + Embeddings)",
]

results_df_tab.columns = column_names
results_df_tab.index = row_names

results_df_tab


 ------------------------------------
Feature specification: x

OLS
R2 Outcome (train): 0.24019421010873732
R2 Outcome (test): 0.029441420379574756
R2 Treatment (train): 0.13403257507137034
R2 Treatment (test): 0.07253793232726424

Boosting
R2 Outcome (train): 0.6598999440133138
R2 Outcome (test): 0.23623157172361342
R2 Treatment (train): 0.5950954446188024
R2 Treatment (test): 0.007236967381715309

 ------------------------------------
Feature specification: x_pca

OLS
R2 Outcome (train): 0.5903960655016508
R2 Outcome (test): 0.20024415204408808
R2 Treatment (train): 0.8153845113504947
R2 Treatment (test): 0.557073382341871

Boosting
R2 Outcome (train): 0.9107073032404274
R2 Outcome (test): 0.2641571556514867
R2 Treatment (train): 0.9893915539011792
R2 Treatment (test): 0.6362481277856005

 ------------------------------------
Feature specification: x_sim

OLS
R2 Outcome (train): 0.5850148754421716
R2 Outcome (test): 0.1923050776120402
R2 Treatment (train): 0.8163070074852854
R2 Trea

,R2 Q Train,R2 Q Test,R2 P Train,R2 P Test
OLS (Tabular),0.240194,0.029441,0.134033,0.072538
Boosting (Tabular),0.659900,0.236232,0.595095,0.007237
OLS (Tabular + PCA),0.590396,0.200244,0.815385,0.557073
Boosting (Tabular + PCA),0.910707,0.264157,0.989392,0.636248
OLS (Tabular + Similarities),0.585015,0.192305,0.816307,0.514603
Boosting (Tabular + Similarities),0.899294,0.250501,0.987059,0.585317
OLS (Tabular + Embeddings),0.657943,0.132849,0.923001,0.567548
Boosting (Tabular + Embeddings),0.932187,0.267303,0.991540,0.646700


Summary

In [14]:
results_df = pd.concat([results_df_tab, results_df_deep], axis=0)
results_df

,R2 Q Train,R2 Q Test,R2 P Train,R2 P Test
OLS (Tabular),0.240194,0.029441,0.134033,0.072538
Boosting (Tabular),0.659900,0.236232,0.595095,0.007237
OLS (Tabular + PCA),0.590396,0.200244,0.815385,0.557073
Boosting (Tabular + PCA),0.910707,0.264157,0.989392,0.636248
OLS (Tabular + Similarities),0.585015,0.192305,0.816307,0.514603
Boosting (Tabular + Similarities),0.899294,0.250501,0.987059,0.585317
OLS (Tabular + Embeddings),0.657943,0.132849,0.923001,0.567548
Boosting (Tabular + Embeddings),0.932187,0.267303,0.991540,0.646700
Deep Time Independent,0.672500,0.149200,0.963900,0.664400
Deep Time Dependent,0.703000,0.332800,0.962600,0.634000


## Performance Diff Data

In [15]:
feature_specifications = ["x", "x_pca", "x_sim", "x_emb"]
results_df_diff_tab = pd.DataFrame()

for feature_specification in feature_specifications:
    print("\n ------------------------------------")
    print(f"Feature specification: {feature_specification}")

    print("\nOLS")
    outcome_reg = sm.OLS(train_dict["y_diff"], train_dict[feature_specification]).fit()
    ols_outcome_train = r2_score(
        train_dict["y_diff"], outcome_reg.predict(train_dict[feature_specification])
    )
    print(f"R2 Outcome (train): {ols_outcome_train}")
    ols_outcome_test = r2_score(
        test_dict["y_diff"], outcome_reg.predict(test_dict[feature_specification])
    )
    print(f"R2 Outcome (test): {ols_outcome_test}")

    treatment_reg = sm.OLS(
        train_dict["d_diff"], train_dict[feature_specification]
    ).fit()
    ols_treatment_train = r2_score(
        train_dict["d_diff"], treatment_reg.predict(train_dict[feature_specification])
    )
    print(f"R2 Treatment (train): {ols_treatment_train}")
    ols_treatment_test = r2_score(
        test_dict["d_diff"], treatment_reg.predict(test_dict[feature_specification])
    )
    print(f"R2 Treatment (test): {ols_treatment_test}")

    print("\nBoosting")
    boost_q_outcome = LGBMRegressor(
        n_estimators=500, lr=0.02, random_state=42, verbose=-1
    )
    boost_q_outcome.fit(train_dict[feature_specification], train_dict["y_diff"])
    r2_boost_q_train = boost_q_outcome.score(
        train_dict[feature_specification], train_dict["y_diff"]
    )
    print(f"R2 Outcome (train): {r2_boost_q_train}")
    r2_boost_q_test = boost_q_outcome.score(
        test_dict[feature_specification], test_dict["y_diff"]
    )
    print(f"R2 Outcome (test): {r2_boost_q_test}")

    boost_q_treatment = LGBMRegressor(
        n_estimators=500, lr=0.02, random_state=42, verbose=-1
    )
    boost_q_treatment.fit(train_dict[feature_specification], train_dict["d_diff"])
    r2_boost_d_train = boost_q_treatment.score(
        train_dict[feature_specification], train_dict["d_diff"]
    )
    print(f"R2 Treatment (train): {r2_boost_d_train}")
    r2_boost_d_test = boost_q_treatment.score(
        test_dict[feature_specification], test_dict["d_diff"]
    )
    print(f"R2 Treatment (test): {r2_boost_d_test}")

    results_df_diff_tab_tmp = pd.DataFrame(
        data=[
            [
                ols_outcome_train,
                ols_outcome_test,
                ols_treatment_train,
                ols_treatment_test,
            ],
            [r2_boost_q_train, r2_boost_q_test, r2_boost_d_train, r2_boost_d_test],
        ],
    )
    results_df_diff_tab = pd.concat(
        [results_df_diff_tab, results_df_diff_tab_tmp], axis=0
    )

row_names = [
    "OLS (Tabular)",
    "Boosting (Tabular)",
    "OLS (Tabular + PCA)",
    "Boosting (Tabular + PCA)",
    "OLS (Tabular + Similarities)",
    "Boosting (Tabular + Similarities)",
    "OLS (Tabular + Embeddings)",
    "Boosting (Tabular + Embeddings)",
]

results_df_diff_tab.columns = column_names
results_df_diff_tab.index = row_names

results_df_diff_tab


 ------------------------------------
Feature specification: x

OLS
R2 Outcome (train): 0.027766630639759926
R2 Outcome (test): 0.02576098400420035
R2 Treatment (train): 0.006245000317056326
R2 Treatment (test): 0.0025641377447166436

Boosting
R2 Outcome (train): 0.38028443423954483
R2 Outcome (test): -0.002089194829201668
R2 Treatment (train): 0.2969335427618205
R2 Treatment (test): -0.07860643983735938

 ------------------------------------
Feature specification: x_pca

OLS
R2 Outcome (train): 0.03119942091650596
R2 Outcome (test): 0.022966601836014844
R2 Treatment (train): 0.006944917178787646
R2 Treatment (test): 0.0027360732084107386

Boosting
R2 Outcome (train): 0.5768266441437295
R2 Outcome (test): 0.03553163157086747
R2 Treatment (train): 0.4630348933951932
R2 Treatment (test): -0.08234115748140769

 ------------------------------------
Feature specification: x_sim

OLS
R2 Outcome (train): 0.031193129078906257
R2 Outcome (test): 0.02468476929146124
R2 Treatment (train): 0.0070

,R2 Q Train,R2 Q Test,R2 P Train,R2 P Test
OLS (Tabular),0.027767,0.025761,0.006245,0.002564
Boosting (Tabular),0.380284,-0.002089,0.296934,-0.078606
OLS (Tabular + PCA),0.031199,0.022967,0.006945,0.002736
Boosting (Tabular + PCA),0.576827,0.035532,0.463035,-0.082341
OLS (Tabular + Similarities),0.031193,0.024685,0.007033,0.003069
Boosting (Tabular + Similarities),0.545594,0.017449,0.431284,-0.140042
OLS (Tabular + Embeddings),0.044330,-0.096100,0.017339,-0.038042
Boosting (Tabular + Embeddings),0.679120,0.052538,0.555512,-0.102137


Summary

In [19]:
results_df_diff = pd.concat([results_df_diff_tab, results_df_diff_deep], axis=0)
results_df_diff

,R2 Q Train,R2 Q Test,R2 P Train,R2 P Test
OLS (Tabular),0.027767,0.025761,0.006245,0.002564
Boosting (Tabular),0.380284,-0.002089,0.296934,-0.078606
OLS (Tabular + PCA),0.031199,0.022967,0.006945,0.002736
Boosting (Tabular + PCA),0.576827,0.035532,0.463035,-0.082341
OLS (Tabular + Similarities),0.031193,0.024685,0.007033,0.003069
Boosting (Tabular + Similarities),0.545594,0.017449,0.431284,-0.140042
OLS (Tabular + Embeddings),0.044330,-0.096100,0.017339,-0.038042
Boosting (Tabular + Embeddings),0.679120,0.052538,0.555512,-0.102137
Deep Time Independent,NaN,NaN,NaN,NaN
Deep Time Dependent,NaN,NaN,NaN,NaN


## Summary

In [20]:
results_df[["R2 Q Test", "R2 P Test"]].round(4) * 100

,R2 Q Test,R2 P Test
OLS (Tabular),2.94,7.25
Boosting (Tabular),23.62,0.72
OLS (Tabular + PCA),20.02,55.71
Boosting (Tabular + PCA),26.42,63.62
OLS (Tabular + Similarities),19.23,51.46
Boosting (Tabular + Similarities),25.05,58.53
OLS (Tabular + Embeddings),13.28,56.75
Boosting (Tabular + Embeddings),26.73,64.67
Deep Time Independent,14.92,66.44
Deep Time Dependent,33.28,63.40


In [21]:
results_df_diff[["R2 Q Test", "R2 P Test"]].round(4) * 100

,R2 Q Test,R2 P Test
OLS (Tabular),2.58,0.26
Boosting (Tabular),-0.21,-7.86
OLS (Tabular + PCA),2.30,0.27
Boosting (Tabular + PCA),3.55,-8.23
OLS (Tabular + Similarities),2.47,0.31
Boosting (Tabular + Similarities),1.74,-14.00
OLS (Tabular + Embeddings),-9.61,-3.80
Boosting (Tabular + Embeddings),5.25,-10.21
Deep Time Independent,NaN,NaN
Deep Time Dependent,NaN,NaN
